# The stages, one at a time

`run_day` is the five stages wired together. This notebook calls each on its
own, so you can see what it takes, what it decides, and what it refuses.

Order follows §5: pickups, processing, line-haul, last mile, returns.

In [ ]:
from datetime import date

from ddn import assumptions, linehaul, pickups, processing, returns
from ddn import solver_adapter as sa
from ddn.allocation import EFFECTIVE_PER_BIKE, allocate, place
from ddn.allocation import vans as van_split
from ddn.lastmile import select
from ddn.model import travel as road
from ddn.solver_adapter import postcheck

HOUR = 3600
TODAY = date(2026, 9, 16)
HUB = {"id": "HUB", "lat": 9.9333, "lon": -84.0833}
from tests.matrices import road_matrix, rows

## §4.2 — allocation, before anything is routed

`docs/solver-capabilities.md` found that a vehicle's depot is fixed before the
solve, so §4.2's "integrated" option cannot be expressed and two-stage is the
only shape available. Allocation is that first stage.

§7.2 makes reallocation between days a cost, so `place` keeps every bike where
it already is and reports how many had to move.

In [ ]:
pools = {"HUB": 1600, "D1": 850, "D2": 650, "D3": 550}
fleet = [f"MOTO-{n:03d}" for n in range(1, 41)]

targets = allocate(pools, len(fleet))
today_plan = place(targets, fleet)
print("bikes per facility:", targets, f"(~{EFFECTIVE_PER_BIKE} envelopes each)")
print("moves on day one:", today_plan.moves)

yesterday = {a.vehicle_id: a.facility_id for a in today_plan.allocations}
shifted = place(allocate({"HUB": 800, "D1": 1700, "D2": 600, "D3": 550}, len(fleet)),
                fleet, previous=yesterday)
print("moves when demand shifts:", shifted.moves, "- not the whole fleet")

§4.3's van split is the contested one: the same fleet covers pickups during the
day and line-haul in the afternoon. §5.1.3 says the earmark should taper.

In [ ]:
split = van_split([f"VAN-{n:02d}" for n in range(1, 11)],
                  taper_from=12 * HOUR, taper_until=15 * HOUR)
for a in split.allocations:
    when = "—" if a.release_at is None else f"released {a.release_at // HOUR}:00"
    print(f"  {a.vehicle_id}  {a.role:<9}{when}")

## §5.1 — pickups, van-only

Two halves. `admission` answers *which van may take this bag*; `dispatch` runs
the day on §5.1.5's cadence, freezing stops already visited and handling
§5.1.7's exceptions.

In [ ]:
bags = [{"mailbag_id": f"BAG-{i}", "customer_id": f"C{i}",
         "lat": 9.9333 + i / 500, "lon": -84.0833 + i / 600,
         "requested_at": 7 * HOUR, "expected_weight_g": 6000,
         "envelope_count": 30} for i in range(6)]
van = {"vehicle_id": "VAN-01", "type": "van", "role": "pickup",
       "capacity_mailbags": assumptions.MAILBAGS_PER_VAN,
       "capacity_weight_g": 500_000, "shift_start": 7 * HOUR,
       "shift_end": 18 * HOUR}

# Real road seconds, replayed from tests/fixtures/road.json.gz.
points = [HUB, *bags]
travel = road.over(road_matrix(points), road.index_of(points))

dispatch = pickups.run(bags, [van], HUB, travel=travel, cut_off=18 * HOUR)
print("collected:", dispatch.collected)
print("back at the hub:", {v: f"{t // HOUR}:{t % HOUR // 60:02d}"
                           for v, t in dispatch.returned_at.items()})

§7.1 makes pickups van-only for security, and the same admission rule that
enforces capacity enforces that.

In [ ]:
bike = dict(van, vehicle_id="MOTO-1", type="motorbike", capacity_mailbags=0)
refused = pickups.run(bags[:1], [bike], HUB, travel=travel, cut_off=18 * HOUR)
print("a motorbike collected:", refused.collected or "nothing")
print("unplaced:", refused.unplaced)

§5.1.7's exceptions. A broken seal is **collected and flagged** — the hub has
to reconcile it, which cannot happen if the bag stays on the doorstep.

In [ ]:
for incident in (pickups.Incident.SEAL_BROKEN, pickups.Incident.SITE_CLOSED,
                 pickups.Incident.BAG_NOT_READY):
    out = pickups.run(bags[:1], [van], HUB, travel=travel, cut_off=18 * HOUR,
                      incidents={"BAG-0": incident})
    visit, = out.visits
    print(f"{incident.value:<24} collected={visit.collected!s:<6}"
          f" flagged={visit.flagged!s:<6} re-request={visit.re_request}")

## §5.2 — processing, which the API only *observes*

§13.4 is explicit that reconciliation, assembly and sorting are physical hub
processes. What the solver needs from them is one number per envelope: when it
will be ready.

§5.2.5's formula is a sum, but the word *queue* matters — "envelopes per hour"
describes a server working through a line, not a delay each envelope suffers
privately. Modelled the other way, the hub would be infinitely parallel and
§8.3's clean-room bottleneck impossible.

In [ ]:
pool = [{"package_id": f"P{i}", "mailbag_id": "BAG-0",
         "package_type": "assembly" if i % 3 == 0 else "finished"}
        for i in range(10)]

ready = processing.schedule(pool, {"BAG-0": 9 * HOUR})
for r in ready[:4]:
    stamp = lambda s: f"{s // HOUR}:{s % HOUR // 60:02d}"
    print(f"  {r.package_id:<4} reconciled {stamp(r.reconciled_at)}"
          f"  assembled {stamp(r.assembled_at) if r.assembled else '—':<6}"
          f"  ready {stamp(r.ready_at)}")
print(f"\nfirst ready {ready[0].ready_at // HOUR}h, "
      f"last {ready[-1].ready_at // HOUR}h — a queue, not ten private delays")

§3.2 asks for zip-centroid envelopes that straddle two facilities to be
flagged. Open Question 14 asks what to *do* about them, so this flags and does
not decide.

In [ ]:
facilities = [{"id": "HUB", "lat": 9.9333, "lon": -84.0833},
              {"id": "D1", "lat": 9.9981, "lon": -84.1197}]
between = {"package_id": "P-mid", "lat": 9.9657, "lon": -84.1015,
           "coord_source": "zip_centroid"}
beside = dict(between, package_id="P-near", lat=9.9333, lon=-84.0833)

# §3.3 assigns by road distance, so the caller supplies a matrix.
pts = [*facilities, between, beside]
sorted_pool = processing.presort([between, beside], facilities,
                                 road_matrix(pts), rows(pts))
for s in sorted_pool:
    print(f"  {s.package_id:<8} -> {s.facility_id:<4} runner-up {s.runner_up}"
          f"  margin {s.margin_m:>8,.0f} m  straddles={s.straddles}")

## §5.3 — line-haul, an assignment rather than a route

§5.3 calls it "primarily an assignment problem; each van serves one depot per
trip". There is nothing to sequence. What has to be decided is which van goes
where, against a deadline that belongs to the destination: the depot's morning
release.

Departing **late** is correct — a van that leaves early strands everything
still in the clean room.

In [ ]:
depots = [{"id": "D1", "route_release_time": 7 * HOUR, "transit_from_hub_min": 30},
          {"id": "D6", "route_release_time": 7 * HOUR, "transit_from_hub_min": 240}]
envelopes = ([{"package_id": f"D1-{i}", "facility_id": "D1",
               "expected_ready_at": 16 * HOUR} for i in range(3)]
             + [{"package_id": f"D6-{i}", "facility_id": "D6",
                 "expected_ready_at": 16 * HOUR} for i in range(2)])

night = linehaul.plan(depots, envelopes, [{"vehicle_id": "VAN-01"},
                                          {"vehicle_id": "VAN-02"}],
                      unload_seconds=assumptions.FACILITY_UNLOAD_MIN * 60)
for trip in night.trips:
    print(f"  {trip.van_id} -> {trip.destination}: departs "
          f"{trip.departure // HOUR}h, arrives {trip.arrival // HOUR}h, "
          f"{len(trip.package_ids)} envelopes")

With one van, the tighter deadline is served first and the other depot rolls —
with a reason, because a fleet too small and a hub too slow are different
problems.

In [ ]:
short = linehaul.plan(depots, envelopes, [{"vehicle_id": "VAN-01"}],
                      unload_seconds=assumptions.FACILITY_UNLOAD_MIN * 60)
print("carried:", short.carried, " held:", short.held)
for facility, reason in short.reasons.items():
    print(f"  {facility}: {reason}")

## §5.4 — last mile, and who is left

§8 says choosing what to leave is "a first-class decision". `select` makes it
before the solver sees the pool, so the answer is explainable: the lowest
priority at a facility short of bikes, rather than whatever the objective
happened to decline.

§6.1's SLA-today and §8.2's locked envelopes are never trimmed.

In [ ]:
pool = [{"package_id": f"P{i:03d}", "priority": float(1000 + i),
         "sla_date": "2026-09-20", "lat": 9.94, "lon": -84.08,
         "status": "Ready", "geocode_confidence": "high"} for i in range(30)]
pool[0] = dict(pool[0], sla_date=TODAY.isoformat())      # §6.1: must go today
pool[1] = dict(pool[1], locked_vehicle_id="MOTO-007")    # §8.2: operations

kept, declined = select(pool, capacity=25, today=TODAY)
print(f"offered {len(kept)}, declined {len(declined)} for want of capacity")
print("declined:", sorted(u.package_id for u in declined))
print("the two that cannot be trimmed are still in:",
      pool[0]["package_id"] in {p['package_id'] for p in kept},
      pool[1]["package_id"] in {p['package_id'] for p in kept})

## §7.1 — every routing result carries its violations

§13.1 requires it and CLAUDE.md makes it an invariant: a violation is a failing
test, never a warning. The post-check delegates to the platform's independent
verifier for what it owns and adds the bullets it cannot see.

In [ ]:
print("§7.1 has", len(sa.BULLETS), "bullets. The first three:")
for bullet in sa.BULLETS[:3]:
    print("  -", bullet)

# Which half a bullet falls in, counted rather than typed into the sentence
# below. It read "seven ... four" until now: §7.1 went to thirteen bullets in
# v0.12 when the two transfer ones arrived, and every copy of the split that
# lived in prose stayed on the pre-v0.12 numbers.
# `tests/test_solver_adapter.py` holds the three inside `postcheck` to this
# same set; this notebook was the copy it could not reach.
day_spanning = {postcheck.ONE_VEHICLE_PER_DAY, postcheck.AFTER_ARRIVAL,
                postcheck.VAN_UNLOADED_FIRST, postcheck.COMBINED_LOAD,
                postcheck.TRANSFER_WITHIN_SLA}
assert day_spanning <= set(sa.BULLETS)

print(f"\n{len(sa.BULLETS) - len(day_spanning)} can be answered from one "
      f"solve; {len(day_spanning)} need the whole day —")
print("an envelope on two vehicles *per day* spans seven facility solves.")

## §5.5 — the return run

§6 sends Rejected and Returned envelopes back; §6.1 adds those whose SLA has
run out. §5.5 groups them by customer site, because several envelopes going
back to one sender is one visit.

In [ ]:
going_back = [{"package_id": f"R{i}", "customer_id": f"C{i % 3}",
               "previous_outcome": "Rejected" if i % 2 else "Returned",
               "facility_id": "HUB", "weight_g": 200,
               "customer_lat": 9.93 + i / 400, "customer_lon": -84.08 + i / 500,
               "sla_date": "2026-09-30"} for i in range(9)]

stops = returns.sites(going_back)
print(f"{len(going_back)} envelopes -> {len(stops)} stops")
for stop in stops:
    print(f"  {stop.customer_id}: {len(stop.package_ids)} envelopes")
print("\nfleet, pending Open Question 13:",
      {v['type'] for v in returns.fleet(['VAN-01'], shift_start=0, shift_end=1)})